In [2]:
!pip install tiktoken 

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken
import urllib.request
import os
import zipfile
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_name = torch.cuda.get_device_name(0)
print(f"Using device: {device}")
print(f"Device name: {device_name}")

Using device: cuda
Device name: Tesla T4


In [4]:
def download_and_extract(url, zip_path):
    if not os.path.exists('cornell movie dialogs corpus'):

        print("Downloading dataset...")
        urllib.request.urlretrieve(url, zip_path)

        print("Extracting dataset...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall()
        print("Dataset downloaded and extracted.")

url = "https://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip"
zip_path = "cornell_movie_dialogs_corpus.zip"
download_and_extract(url, zip_path)


Extracting dataset...
Dataset downloaded and extracted.


In [34]:
lines_filepath = "cornell movie-dialogs corpus/movie_lines.txt"
dialogues = []
with open(lines_filepath, 'r', encoding = 'iso-8859-1') as f:
    for line in f:
        parts = line.strip().split(" +++$+++ ")
        if len(parts) == 5:
            dialogues.append(parts[4].strip())

raw_text = '\n'.join(dialogues)

print(f"Total characters in dataset: {len(raw_text)}")
print(f"Sample text: {raw_text[:500]}")

Total characters in dataset: 17146310
Sample text: They do not!
They do to!
I hope so.
She okay?
Let's go.
Wow
Okay -- you're gonna need to learn how to lie.
No
I'm kidding.  You know how sometimes you just become this "persona"?  And you don't know how to quit?
Like my fear of wearing pastels?
The "real you".
What good stuff?
I figured you'd get to the good stuff eventually.
Thank God!  If I had to hear one more story about your coiffure...
Me.  This endless ...blonde babble. I'm like, boring myself.
What crap?
do you listen to this crap?
No...


In [39]:
class MovieDialogueDataset(Dataset):
    def __init__(self, text, tokenizer, block_size, stride=None):
        self.tokens = tokenizer.encode(text)
        self.block_size = block_size
        self.stride = stride if stride is not None else block_size
        # all valid starting indices
        self.indices = list(range(0, len(self.tokens) - block_size, self.stride))
        print(f"Total tokens: {len(self.tokens)}")
        print(f"Total samples: {len(self.indices)}")

    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        start = self.indices[idx]
        chunk = self.tokens[start : start + self.block_size + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y
    
block_size = 128
batch_size = 32
stride = 128

tokenizer = tiktoken.get_encoding('gpt2')

dataset = MovieDialogueDataset(raw_text, tokenizer, block_size, stride)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

Total tokens: 4790644
Total samples: 37426


In [36]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim = True) + self.eps) # faster than sqrt and then dividing
        return x * norm * self.weight
    

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        return self.net(x)
    

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size, dropout = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer(
            'mask', torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.shape   # batch size, seq length, embedding dim

        q, k, v = self.qkv_proj(x).split(C, dim=2)

        # divide into heads and reshape into B, H, T, D for multi-head computation
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale    # B, H, T, T
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        attn = torch.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        
        out = attn @ v  # B, H, T, D
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)
    

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_head, block_size, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_head, block_size, dropout)
        self.norm2 = RMSNorm(embed_dim)
        self.ff = FeedForward(embed_dim, hidden_dim= 4 * embed_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x
    

class GPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_head, num_layer, block_size, dropout = 0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.positon_embedding = nn.Embedding(block_size, embed_dim)
        self.blocks = nn.Sequential(
            *[TransformerBlock(embed_dim, num_head, block_size) for _ in range(num_layer)]
        )
        self.norm = RMSNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.block_size = block_size

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.positon_embedding(pos)
        x = self.blocks(x)
        logits = self.lm_head(self.norm(x))
        return logits
    

vocab_size = tokenizer.n_vocab
embed_dim = 128
num_heads = 4 
num_layers = 4  
dropout = 0.1

model = GPT(vocab_size, embed_dim, num_heads, num_layers, block_size, dropout).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params / 1e6:.2f}M")

Model parameters: 13.67M


In [41]:
lr = 3e-4
epochs = 5

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

total_steps = epochs * len(dataloader)
warmup_steps = int(0.1 * total_steps)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr = lr,
    total_steps = total_steps,
    pct_start = 0.1
)

# model computes in FP16 with gradients in FP32
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

In [42]:
model.train()

for epoch in range(epochs):
    total_loss = 0.0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}")

    for step, (x, y) in enumerate(progress_bar):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(x)
            B, T, V = logits.shape
            loss = nn.functional.cross_entropy(
                logits.view(B * T, V),
                y.view(B * T)
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        avg_loss = total_loss / (step + 1)

        progress_bar.set_postfix({
            'loss': f'{avg_loss:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.6f}'
        })

    print(f">>> Epoch {epoch+1} complete | Avg Loss: {total_loss / len(dataloader):.4f}\n")

print("Training complete!")

Epoch 1/5: 100%|██████████| 1170/1170 [01:16<00:00, 15.37it/s, loss=5.7624, lr=0.000291]


>>> Epoch 1 complete | Avg Loss: 5.7624



Epoch 2/5: 100%|██████████| 1170/1170 [01:15<00:00, 15.45it/s, loss=4.8371, lr=0.000225]


>>> Epoch 2 complete | Avg Loss: 4.8371



Epoch 3/5: 100%|██████████| 1170/1170 [01:15<00:00, 15.48it/s, loss=4.6381, lr=0.000124]


>>> Epoch 3 complete | Avg Loss: 4.6381



Epoch 4/5: 100%|██████████| 1170/1170 [01:15<00:00, 15.48it/s, loss=4.5340, lr=0.000035]


>>> Epoch 4 complete | Avg Loss: 4.5340



Epoch 5/5: 100%|██████████| 1170/1170 [01:15<00:00, 15.45it/s, loss=4.4951, lr=0.000000]

>>> Epoch 5 complete | Avg Loss: 4.4951

Training complete!


In [43]:
model.eval()

def generate(model, tokenizer, prompt, max_new_tokens=200, temperature=1.0, top_k=50):
    tokens = tokenizer.encode(prompt)
    idx = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)    # (1, T)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -model.block_size: ]
            logits = model(idx_cond)    # (1, T, vocab_size)
            logits = logits[:, -1, :]
            logits = logits / temperature

            top_k_logits, _ = torch.topk(logits, top_k, dim=-1)
            min_val = top_k_logits[:, -1].unsqueeze(-1)
            logits = logits.masked_fill(logits < min_val, float('-inf'))

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)    # (1, 1)
            idx = torch.cat([idx, next_token], dim=1)   # (1, T+1)

    generated_tokens = idx[0].tolist()
    return tokenizer.decode(generated_tokens)


prompt = "I love you"

output = generate(
        model,
        tokenizer,
        prompt,
        max_new_tokens=150,
        temperature=0.8,
        top_k=40
    )

print(output)

I love you with you.
I'm okay.
I'll see you to make a lot of your ass from the first, but it's. You're all these of those of your name.
My God.
I have to come to me --  My old.
It's nothing of me.
So, I'm not sure you're not sure, but you don't know why you go out. But what it'll be too.
I've got to be a few minutes.
Just be a girl.
It's a great, there's just a girl --
Well, I can do that you're not sure.  He didn't leave me.  Can I say.
I do it.  I do
